# RoadVision AI - Dataset Cleanning

#### Handle corrupted images, missing or invalid annotations, duplicates, and other confirmed data-quality problems.

### Imports and paths

In [1]:
from pathlib import Path
from collections import defaultdict, Counter
import hashlib
import math
import shutil

import pandas as pd
from PIL import Image, UnidentifiedImageError
import yaml

DATASET_VERSION = Path(r"C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2")
ROOT = DATASET_VERSION / "trafic_data"

OUTPUT_ROOT = DATASET_VERSION / "roadvision_cleaned"
REPORT_DIR = OUTPUT_ROOT / "reports"

SPLITS = ["train", "valid", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

print("Original dataset:", ROOT.resolve())
print("Original exists:", ROOT.exists())
print("Clean output:", OUTPUT_ROOT.resolve())
if not ROOT.is_dir():
    raise FileNotFoundError(f"Dataset root not found: {ROOT}. Update DATASET_VERSION to your actual dataset version folder.")
if OUTPUT_ROOT.resolve() == ROOT.resolve() or ROOT.resolve() in OUTPUT_ROOT.resolve().parents:
    raise ValueError("Safety check failed: output must be outside the original dataset root.")

Original dataset: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\trafic_data
Original exists: True
Clean output: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\roadvision_cleaned


### Load class configuration

In [2]:
yaml_path = ROOT / "data_1.yaml"
if not yaml_path.is_file():
    raise FileNotFoundError(f"Expected YOLO config not found: {yaml_path}")

with yaml_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f) or {}

class_names = config.get("names", [])
if isinstance(class_names, dict):
    class_names = [class_names[k] for k in sorted(class_names, key=lambda x: int(x))]
nc = int(config.get("nc", len(class_names)))
if class_names and len(class_names) != nc:
    raise ValueError(f"YAML mismatch: nc={nc}, but there are {len(class_names)} class names")

print("Number of classes:", nc)
for i, name in enumerate(class_names):
    print(f"{i:2} -> {name}")

Number of classes: 21
 0 -> ambulance
 1 -> army vehicle
 2 -> auto rickshaw
 3 -> bicycle
 4 -> bus
 5 -> car
 6 -> garbagevan
 7 -> human hauler
 8 -> minibus
 9 -> minivan
10 -> motorbike
11 -> pickup
12 -> policecar
13 -> rickshaw
14 -> scooter
15 -> suv
16 -> taxi
17 -> three wheelers -CNG-
18 -> truck
19 -> van
20 -> wheelbarrow


### Verify image integrity and detect exact duplicates

In [3]:
image_integrity_rows = []
hash_to_images = defaultdict(list)

for split in SPLITS:
    image_dir = ROOT / split / "images"
    if not image_dir.is_dir():
        print(f"Skipping {split}: images folder not found")
        continue
    for image_path in sorted(image_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        status, error, digest = "ok", "", None
        try:
            with Image.open(image_path) as im:
                im.verify()
            # Re-open and force decoding; verify() alone does not decode all pixel data.
            with Image.open(image_path) as im:
                im.load()
                if im.width <= 0 or im.height <= 0:
                    raise ValueError("Image has zero width or height")
            digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
            hash_to_images[digest].append((split, image_path))
        except (UnidentifiedImageError, OSError, ValueError, SyntaxError) as exc:
            status, error = "corrupted", str(exc)
        image_integrity_rows.append({
            "split": split, "image": str(image_path), "status": status,
            "error": error, "sha256": digest
        })

image_integrity_df = pd.DataFrame(image_integrity_rows)
corrupted_df = image_integrity_df[image_integrity_df["status"] == "corrupted"].copy()
duplicate_rows = []
for digest, entries in hash_to_images.items():
    if len(entries) > 1:
        for split, path in entries:
            duplicate_rows.append({"sha256": digest, "split": split, "image": str(path), "duplicate_count": len(entries)})
duplicates_df = pd.DataFrame(duplicate_rows, columns=["sha256", "split", "image", "duplicate_count"])
print("Images checked:", len(image_integrity_df))
print("Corrupted images:", len(corrupted_df))
print("Images participating in exact-duplicate groups:", len(duplicates_df))
display(corrupted_df.head(20))
display(duplicates_df.head(30))

Skipping test: images folder not found
Images checked: 3004
Corrupted images: 0
Images participating in exact-duplicate groups: 2


,split,image,status,error,sha256


,sha256,split,image,duplicate_count
0,2ee1490b44e2afebab03e55838fe623aa31d99074bbf06...,train,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,2
1,2ee1490b44e2afebab03e55838fe623aa31d99074bbf06...,train,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,2


### Validate annotations and prepare cleaned label text

In [4]:
def validate_yolo_line(line):
    """Return (is_valid, reason, normalized_line, class_id)."""
    parts = line.split()
    if len(parts) != 5:
        return False, "wrong_number_of_values", None, None
    try:
        class_float = float(parts[0])
        if not math.isfinite(class_float) or not class_float.is_integer():
            return False, "class_id_not_integer", None, None
        class_id = int(class_float)
        values = [float(x) for x in parts[1:]]
    except (TypeError, ValueError):
        return False, "non_numeric_value", None, None
    if not all(math.isfinite(x) for x in values):
        return False, "non_finite_value", None, class_id
    if not (0 <= class_id < nc):
        return False, "class_id_out_of_range", None, class_id
    x_center, y_center, width, height = values
    if not (0 <= x_center <= 1 and 0 <= y_center <= 1):
        return False, "center_out_of_range", None, class_id
    if not (0 < width <= 1 and 0 < height <= 1):
        return False, "non_positive_or_oversized_box", None, class_id
    x_min, x_max = x_center - width / 2, x_center + width / 2
    y_min, y_max = y_center - height / 2, y_center + height / 2
    if x_min < 0 or x_max > 1 or y_min < 0 or y_max > 1:
        return False, "box_outside_image_bounds", None, class_id
    clean_line = str(class_id) + " " + " ".join(f"{x:.8g}" for x in values)
    return True, "", clean_line, class_id

label_audit_rows = []
label_file_rows = []
valid_label_lines = {}
class_counts_clean = Counter()

for split in SPLITS:
    label_dir = ROOT / split / "labels"
    if not label_dir.is_dir():
        continue
    for label_path in sorted(label_dir.glob("*.txt")):
        raw_lines = label_path.read_text(encoding="utf-8", errors="replace").splitlines()
        kept_lines = []
        invalid_count = 0
        for line_no, raw_line in enumerate(raw_lines, start=1):
            if not raw_line.strip():
                continue
            ok, reason, clean_line, class_id = validate_yolo_line(raw_line)
            if ok:
                kept_lines.append(clean_line)
                class_counts_clean[class_id] += 1
            else:
                invalid_count += 1
                label_audit_rows.append({
                    "split": split, "label_file": str(label_path), "line_number": line_no,
                    "reason": reason, "original_line": raw_line
                })
        valid_label_lines[(split, label_path.stem)] = kept_lines
        label_file_rows.append({
            "split": split, "label_file": str(label_path),
            "original_nonempty_lines": sum(bool(x.strip()) for x in raw_lines),
            "valid_lines_kept": len(kept_lines), "invalid_lines_removed_in_copy": invalid_count,
            "empty_after_cleaning": len(kept_lines) == 0
        })

label_audit_df = pd.DataFrame(label_audit_rows, columns=["split", "label_file", "line_number", "reason", "original_line"])
label_summary_df = pd.DataFrame(label_file_rows)
print("Label files checked:", len(label_summary_df))
print("Invalid annotation lines flagged:", len(label_audit_df))
print("Label files empty after cleaning:", int(label_summary_df["empty_after_cleaning"].sum()) if not label_summary_df.empty else 0)
display(label_audit_df.head(50))

Label files checked: 3004
Invalid annotation lines flagged: 24
Label files empty after cleaning: 4


,split,label_file,line_number,reason,original_line
0,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,1,non_positive_or_oversized_box,9 0 0 0 0
1,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,2,non_positive_or_oversized_box,15 0 0 0 0
2,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,3,non_positive_or_oversized_box,13 0 0 0 0
3,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,4,non_positive_or_oversized_box,13 0 0 0 0
4,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,5,non_positive_or_oversized_box,13 0 0 0 0
5,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,6,non_positive_or_oversized_box,13 0 0 0 0
6,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,7,non_positive_or_oversized_box,13 0 0 0 0
7,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,8,non_positive_or_oversized_box,13 0 0 0 0
8,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,9,non_positive_or_oversized_box,13 0 0 0 0
9,valid,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,10,non_positive_or_oversized_box,13 0 0 0 0


### Build the cleaned copy 

In [5]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
copy_log_rows = []
corrupted_paths = set(corrupted_df["image"].tolist()) if not corrupted_df.empty else set()

for split in SPLITS:
    src_img_dir = ROOT / split / "images"
    src_lbl_dir = ROOT / split / "labels"
    dst_img_dir = OUTPUT_ROOT / split / "images"
    dst_lbl_dir = OUTPUT_ROOT / split / "labels"
    if not src_img_dir.is_dir():
        continue
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    for image_path in sorted(src_img_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        if str(image_path) in corrupted_paths:
            copy_log_rows.append({"split": split, "file": str(image_path), "action": "excluded_corrupted_image"})
            continue
        shutil.copy2(image_path, dst_img_dir / image_path.name)
        label_path = src_lbl_dir / f"{image_path.stem}.txt"
        if label_path.is_file():
            kept = valid_label_lines.get((split, label_path.stem), [])
            (dst_lbl_dir / label_path.name).write_text("\n".join(kept) + ("\n" if kept else ""), encoding="utf-8")
            action = "copied_image_and_cleaned_label"
        else:
            # Pairing validation previously found no missing labels; keep an audit entry if this changes.
            action = "copied_image_without_label"
        copy_log_rows.append({"split": split, "file": str(image_path), "action": action})

# Copy the original YAML as a reference; split paths are intentionally not rewritten in this notebook.
shutil.copy2(yaml_path, OUTPUT_ROOT / yaml_path.name)
copy_log_df = pd.DataFrame(copy_log_rows)
print("Cleaned copy created at:", OUTPUT_ROOT.resolve())
print("Files/actions logged:", len(copy_log_df))
display(copy_log_df["action"].value_counts().rename_axis("action").reset_index(name="count"))

Cleaned copy created at: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\roadvision_cleaned
Files/actions logged: 3004


,action,count
0,copied_image_and_cleaned_label,3004


### Export cleaning reports

In [6]:
image_integrity_df.to_csv(REPORT_DIR / "image_integrity.csv", index=False)
corrupted_df.to_csv(REPORT_DIR / "corrupted_images.csv", index=False)
duplicates_df.to_csv(REPORT_DIR / "exact_duplicate_images.csv", index=False)
label_audit_df.to_csv(REPORT_DIR / "invalid_annotation_lines.csv", index=False)
label_summary_df.to_csv(REPORT_DIR / "label_cleaning_summary.csv", index=False)
copy_log_df.to_csv(REPORT_DIR / "cleaning_actions.csv", index=False)

class_distribution_clean = pd.DataFrame([
    {"class_id": i, "class_name": class_names[i] if i < len(class_names) else "unknown",
     "valid_boxes_in_cleaned_labels": class_counts_clean.get(i, 0)}
    for i in range(nc)
])
class_distribution_clean.to_csv(REPORT_DIR / "cleaned_class_distribution.csv", index=False)
display(class_distribution_clean)
print("\nReports saved in:", REPORT_DIR.resolve())
for p in sorted(REPORT_DIR.glob("*.csv")):
    print("-", p.name)

,class_id,class_name,valid_boxes_in_cleaned_labels
0,0,ambulance,70
1,1,army vehicle,43
2,2,auto rickshaw,372
3,3,bicycle,459
4,4,bus,3332
5,5,car,5474
6,6,garbagevan,3
7,7,human hauler,169
8,8,minibus,95
9,9,minivan,931



Reports saved in: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\roadvision_cleaned\reports
- cleaned_class_distribution.csv
- cleaning_actions.csv
- corrupted_images.csv
- exact_duplicate_images.csv
- image_integrity.csv
- invalid_annotation_lines.csv
- label_cleaning_summary.csv
